# Fine-tuning Gemma 2 2B IT

This notebook adapts Gemma for French educational questions. The current dataset has 150 training, 5 validation, and 10 test examples. The latest run completed one epoch and 19 steps. The epoch log showed a training loss of 1.732579 and validation loss of 1.917565; the final trainer summary reported an aggregate training loss of 2.051600 and a runtime of 114.8 seconds. An earlier run used a separate 40-example dataset.

In [ ]:
%pip install -q -U unsloth trl datasets peft accelerate bitsandbytes

## Hugging Face login

In [ ]:
from huggingface_hub import login

login()

## Upload data and save the adapter

Upload `train.jsonl`, `validation.jsonl` and `test.jsonl`. The adapter is saved to Google Drive.

In [ ]:
from google.colab import drive, files

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

required_files = {'train.jsonl', 'validation.jsonl', 'test.jsonl'}
missing_files = required_files.difference(uploaded.keys())
if missing_files:
    raise FileNotFoundError(f'Missing required files: {sorted(missing_files)}')

In [ ]:
from datasets import load_dataset

train_path = '/content/train.jsonl'
validation_path = '/content/validation.jsonl'
test_path = '/content/test.jsonl'

raw_dataset = load_dataset('json', data_files=train_path, split='train')
raw_validation = load_dataset('json', data_files=validation_path, split='train')
raw_test = load_dataset('json', data_files=test_path, split='train')
print('train:', raw_dataset)
print('validation:', raw_validation)
print('test:', raw_test)
print(raw_dataset[0])

## Check the conversations

The target answers use the same three-part format. No tutor prompt is added to the questions.

In [ ]:
print(raw_dataset[0])

In [ ]:
from unsloth import FastLanguageModel

model_name = 'google/gemma-2-2b-it'
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

def format_conversation(example):
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

def make_text_dataset(dataset):
    return dataset.map(
        format_conversation,
        remove_columns=dataset.column_names,
    )

train_dataset = make_text_dataset(raw_dataset)
validation_dataset = make_text_dataset(raw_validation)
test_dataset = make_text_dataset(raw_test)

print(train_dataset[0]['text'])
print('Prepared examples:', len(train_dataset), len(validation_dataset), len(test_dataset))

## LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## Training

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

output_dir = '/content/gemma-2-2b-it-lora-project'

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=1,
    save_strategy='epoch',
    eval_strategy='epoch',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_length=max_seq_length,
    dataset_text_field='text',
    packing=False,
    report_to='none',
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
)

trainer.train()

## Save the adapter


In [ ]:
adapter_dir = '/content/drive/MyDrive/gemma-2-2b-it-lora-final'
import os
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Saved adapter to {adapter_dir}')

## Inference

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [{
    'role': 'user',
    'content': "Qu'est-ce qu'une variable en programmation ?"
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

with torch.inference_mode():
    generated = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
    )

new_tokens = generated[0, inputs.shape[1]:]
output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(output)

## Test data

The test examples are kept separate from training.

In [ ]:
print(test_dataset[0]['text'])
print(f'Test examples available for later evaluation: {len(test_dataset)}')

## Try more questions

This cell uses the model already loaded in the runtime.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompts = [
    "Je ne comprends pas les fractions. Peux-tu m'aider progressivement ?",
    "Explique-moi la photosynthèse comme si j'avais dix ans.",
    "J'ai répondu que 2 + 3 × 4 = 20. Est-ce correct ?",
    "Qu'est-ce qu'une variable en programmation ? Donne une explication simple.",
    "Explique-moi le sujet.",
    "Explique la photosynthèse avec un peu de rigueur scientifique.",
    "Donne-moi un exercice simple sur les pourcentages, mais ne donne pas immédiatement la réponse.",
    "Peux-tu m'expliquer la différence entre un fait et une opinion avec des exemples ?",
]

def generate_tutor_answer(question):
    messages = [{
        'role': 'user',
        'content': question,
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    with torch.inference_mode():
        generated = model.generate(
            input_ids=inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )

    return tokenizer.decode(
        generated[0, inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

for index, question in enumerate(prompts, start=1):
    print('\n' + '=' * 80)
    print(f'TEST {index}: {question}')
    print(generate_tutor_answer(question))

## Compare the base and fine-tuned models

Question utilisée : « Qu’est-ce qu’une variable ? » Aucune consigne de tutorat n’est ajoutée.

In [ ]:
import gc
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompts = [
    "Qu’est-ce qu’une variable ?",
]

def make_inputs(current_tokenizer, question):
    messages = [{
        'role': 'user',
        'content': question,
    }]
    return current_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

def generate_comparison_answer(current_model, current_tokenizer, question):
    inputs = make_inputs(current_tokenizer, question)
    with torch.inference_mode():
        generated = current_model.generate(
            input_ids=inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )
    return current_tokenizer.decode(
        generated[0, inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

after_answers = [
    generate_comparison_answer(model, tokenizer, question)
    for question in prompts
]

del model
gc.collect()
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-2-2b-it',
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

before_answers = [
    generate_comparison_answer(base_model, base_tokenizer, question)
    for question in prompts
]

for index, question in enumerate(prompts, start=1):
    print('\n' + '=' * 80)
    print(f'TEST {index}: {question}')
    print(f'\n--- BEFORE FINE-TUNING ---\n{before_answers[index - 1]}')
    print(f'\n--- AFTER FINE-TUNING ---\n{after_answers[index - 1]}')